<a href="https://colab.research.google.com/github/BEMiklos/PlantTraits2024/blob/main/XGB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q kaggle

In [ ]:
!mkdir ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle competitions download -c planttraits2024

In [ ]:
!unzip -q '/content/planttraits2024.zip'

In [ ]:
import random

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import albumentations as A
import cv2

import torch
from torch import nn
import torch.nn.functional as F
from torchvision.io import read_image
from torch.utils.data import Dataset, ConcatDataset, DataLoader, Subset, random_split
from torchvision import transforms

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import r2_score
from sklearn.preprocessing import PolynomialFeatures

import xgboost as xgb

import re

In [ ]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
random_state = 7

np.random.seed(0)

random.seed(0)

torch.manual_seed(0)
torch.cuda.manual_seed(0)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
TRAIN_IMAGES_PATH = '/content/train_images'
TEST_IMAGES_PATH = '/content/test_images'

### Untility functions

In [ ]:
mean_pattern = r'X\d+_mean'
sd_pattern = r'X\d+_sd'

In [ ]:
def find_matching_elements(lst,pattern):
    matching_elements = [element for element in lst if re.search(pattern, element)]
    return matching_elements

In [ ]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

In [ ]:
train_df.sample(10)

In [ ]:
len(train_df.columns), len(test_df.columns)

In [ ]:
len(train_df), len(test_df)

In [ ]:
targets = [col for col in train_df.columns if col not in test_df.columns]
targets = targets[:6]

In [ ]:
metadata_columns = [col for col in train_df.columns if col in test_df.columns and col != 'id']

 TODO: hi?nyz? ?rt?kek kezel?s?nek pontos?t?sa

In [ ]:
# Check for missing values in the train DataFrame
train_missing_values = train_df.isnull().sum()
print("Missing values in Train DataFrame:")
print(train_missing_values)

In [ ]:
train_df.fillna(train_df.mean(), inplace=True)

In [ ]:
columns_to_log = ['X11_mean', 'X18_mean', 'X26_mean', 'X50_mean', 'X3112_mean',
                  'X4_sd', 'X11_sd', 'X18_sd', 'X26_sd', 'X50_sd', 'X3112_sd']

# Apply log10 transformation to the specified columns
for column in columns_to_log:
    train_df[column] = np.log10(train_df[column] + 1)  # Adding 1 to avoid log(0)


In [ ]:
def split_dataframe(train_df, test_df, train_size=0.8, val_size=0.2):
    if train_size + val_size != 1:
        raise ValueError("train_size, val_size, and test_size must sum to 1")

    train_df, val_df = train_test_split(train_df, train_size=train_size, random_state=random_state)
    return train_df, val_df, test_df

In [ ]:
def r2_loss(output, target):
    target = target
    output = output
    target_mean = torch.mean(target)
    ss_tot = torch.sum((target - target_mean) ** 2)
    ss_res = torch.sum((target - output) ** 2)
    r2 = 1 - ss_res / ss_tot
    return 1 - r2

In [ ]:
!pip -q install torchmetrics

In [ ]:
from torchmetrics.regression import R2Score

# Data preparation

In [ ]:
bemeno_valtozok = metadata_columns

In [ ]:
celvaltozok = targets[:6]
print("Targets:",celvaltozok)

In [ ]:
train_df, val_df, test_df = split_dataframe(train_df.sample(frac=1, random_state=1), test_df.sample(frac=1, random_state=1))

In [ ]:
train_df

# Feature engineering

In [ ]:
worldclim = train_df.columns[1:7]
soil = train_df.columns[7:68]
modis = train_df.columns[68:128]
vod = train_df.columns[128:164]

In [ ]:
targets = ['X4_mean', 'X11_mean', 'X18_mean', 'X26_mean', 'X50_mean','X3112_mean', 'X4_sd',
 'X11_sd',
 'X18_sd',
 'X26_sd',
 'X50_sd',
 'X3112_sd']

In [ ]:
feature_cols = [col for col in train_df.columns if col not in targets]
features = train_df[feature_cols]

In [ ]:
targets = ['X4_mean', 'X11_mean', 'X18_mean', 'X26_mean', 'X50_mean','X3112_mean']

In [ ]:
imputer = SimpleImputer(strategy='median')

features_imputed = imputer.fit_transform(features)
features_imputed = pd.DataFrame(features_imputed, columns=features.columns)

In [ ]:
scaler = StandardScaler()

features_scaled = scaler.fit_transform(features_imputed)
features_scaled = pd.DataFrame(features_scaled, columns=features.columns)

In [ ]:
y_multi = train_df[targets]
X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(
    features_scaled[feature_cols], y_multi, test_size=0.2, random_state=42)

multi_xgb_model = MultiOutputRegressor(xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100, random_state=42))

multi_xgb_model.fit(X_train_multi, y_train_multi)

y_pred_multi = multi_xgb_model.predict(X_test_multi)

r2_scores = r2_score(y_test_multi, y_pred_multi, multioutput='raw_values')
print("R? scores for each target (on scaled features):", r2_scores)

## Dimension reduction based on importance

In [ ]:
X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(features_scaled, train_df[targets], test_size=0.2, random_state=42)

xgb_model_full = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100, random_state=42)
xgb_model_full.fit(X_train_full, y_train_full)

importances = xgb_model_full.feature_importances_

feature_names = features_scaled.columns
importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
importance_df = importance_df.sort_values(by='Importance', ascending=False).reset_index(drop=True)
print(importance_df.head(20))

In [ ]:
top_features = importance_df['Feature'].head(30).tolist()
y_multi = train_df[targets]
X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(
    features_scaled[top_features], y_multi, test_size=0.2, random_state=42)

multi_xgb_model = MultiOutputRegressor(xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100, random_state=42))

multi_xgb_model.fit(X_train_multi, y_train_multi)

y_pred_multi = multi_xgb_model.predict(X_test_multi)

r2_scores = r2_score(y_test_multi, y_pred_multi, multioutput='raw_values')
print("R? scores for each target (with the top 30 important features):", r2_scores)

R? scores for each target (on scaled features): [ 0.06519863,  0.16375705,  0.3071952,   0.23258292, -0.24942554,  0.25492991]

R? scores for each target (with the top 30 important features): [ 0.093593,    0.15704444,  0.303049,    0.228631,   -0.18055592,  0.24359068]

# Dimension reduction with PCA

In [ ]:
pca = PCA(n_components=min(features_scaled.shape))

pca.fit(features_scaled)

cumulative_variance_ratio = np.cumsum(pca.explained_variance_ratio_)

In [ ]:
n_components_95 = np.where(cumulative_variance_ratio >= 0.95)[0][0] + 1

pca_95 = PCA(n_components=n_components_95)

features_pca_95 = pca_95.fit_transform(features_scaled)

features_pca_95_df = pd.DataFrame(features_pca_95, columns=[f'PC{i+1}' for i in range(n_components_95)])
features_pca_95_df.head(), n_components_95

In [ ]:
X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(
    features_pca_95_df, y_multi, test_size=0.2, random_state=42)

multi_xgb_model = MultiOutputRegressor(xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100, random_state=42))

multi_xgb_model.fit(X_train_multi, y_train_multi)

y_pred_multi = multi_xgb_model.predict(X_test_multi)

r2_scores = r2_score(y_test_multi, y_pred_multi, multioutput='raw_values')
print("R? scores for each target (PCA):", r2_scores)

R? scores for each target (on scaled features): [ 0.06519863, 0.16375705, 0.3071952, 0.23258292, -0.24942554, 0.25492991]

R? scores for each target (with the top 30 important features): [ 0.093593, 0.15704444, 0.303049, 0.228631, -0.18055592, 0.24359068]

R? scores for each target (PCA): [ 0.0397597,   0.15494675,  0.28841653,  0.21596206, -0.2079703,   0.24695846]

In [ ]:
X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(features_pca_95_df, train_df[targets], test_size=0.2, random_state=42)

xgb_model_full = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100, random_state=42)
xgb_model_full.fit(X_train_full, y_train_full)

importances = xgb_model_full.feature_importances_

feature_names = features_pca_95_df.columns
importance_df_pca = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
importance_df_pca = importance_df.sort_values(by='Importance', ascending=False).reset_index(drop=True)
print(importance_df_pca.head(20))

In [ ]:
top_features = importance_df_pca['Feature'].head(30).tolist()
y_multi = train_df[targets]
X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(
    features_scaled[top_features], y_multi, test_size=0.2, random_state=42)

multi_xgb_model = MultiOutputRegressor(xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100, random_state=42))

multi_xgb_model.fit(X_train_multi, y_train_multi)

y_pred_multi = multi_xgb_model.predict(X_test_multi)

r2_scores = r2_score(y_test_multi, y_pred_multi, multioutput='raw_values')
print("R? scores for each target (PCA with the top 30 important features):", r2_scores)

R? scores for each target (on scaled features): [ 0.06519863, 0.16375705, 0.3071952, 0.23258292, -0.24942554, 0.25492991]

R? scores for each target (with the top 30 important features): [ 0.093593, 0.15704444, 0.303049, 0.228631, -0.18055592, 0.24359068]

R? scores for each target (PCA): [ 0.0397597, 0.15494675, 0.28841653, 0.21596206, -0.2079703, 0.24695846]

R? scores for each target (PCA with the top 30 important features): [ 0.093593,    0.15704444,  0.303049,    0.228631,   -0.18055592,  0.24359068]

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
top_features = importance_df_pca['Feature'].head(30).tolist()
targets = ['X4_mean', 'X11_mean', 'X18_mean', 'X26_mean', 'X50_mean','X3112_mean']
y_multi = train_df[targets]
X = features_scaled[top_features]

X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(
    X, y_multi, test_size=0.2, random_state=42)

xgb_base = xgb.XGBRegressor(objective='reg:squarederror', random_state=42)

multi_xgb = MultiOutputRegressor(xgb_base)

param_grid = {
    'estimator__n_estimators': [50, 100, 200],
    'estimator__max_depth': [3, 5, 7],
    'estimator__learning_rate': [0.01, 0.1, 0.2]
}

grid_search = GridSearchCV(estimator=multi_xgb, param_grid=param_grid,
                           cv=3, n_jobs=-1, verbose=2, scoring='r2')

grid_search.fit(X_train_multi, y_train_multi)

print("Best parameters found: ", grid_search.best_params_)
print("Best R? score found: ", grid_search.best_score_)

In [ ]:
top_features = importance_df_pca['Feature'].head(30).tolist()
targets = ['X4_mean', 'X11_mean', 'X18_mean', 'X26_mean', 'X50_mean','X3112_mean']
y_multi = train_df[targets]
X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(
    features_scaled[top_features], y_multi, test_size=0.2, random_state=42)

multi_xgb_model = MultiOutputRegressor(xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100, random_state=42, learning_rate=0.1, max_depth=5))

multi_xgb_model.fit(X_train_multi, y_train_multi)

y_pred_multi = multi_xgb_model.predict(X_test_multi)

r2_scores = r2_score(y_test_multi, y_pred_multi, multioutput='raw_values')
print("R? scores for each target (PCA with the top 30 important features):", r2_scores)

R? scores for each target (PCA with the top 30 important features optimized): [0.12358511, 0.17924043, 0.31857293, 0.24550138, 0.01399558, 0.26230538]

 R? scores for each target (PCA with the top 30 important features): [ 0.093593, 0.15704444, 0.303049, 0.228631, -0.18055592, 0.24359068]

